In [1]:
# Notebook: nb_gold_master_view
# Purpose: Create a single, wide table for Power BI (Push-down architecture)
from pyspark.sql import functions as F
from datetime import datetime

print(" Creating Gold Master View for Power BI...")
print("=" * 60)

# 1. Load Gold tables
risk_df = spark.table("gold_company_risk_scores")
sanctions_df = spark.table("gold_sanctions_screening")

print(f" Risk records: {risk_df.count()}")
print(f" Sanctions records: {sanctions_df.count()}")

# 2. Left join Risk Scores with Sanctions Details
# We use a left join so we keep ALL companies, even those without sanctions hits
master_view_df = risk_df.alias("r").join(
    sanctions_df.alias("s"),
    F.col("r.company_registration") == F.col("s.company_registration"),
    "left"
)

# 3. Select and rename final columns for the dashboard
# We only want the most relevant columns to keep the Power BI model clean
final_view_df = master_view_df.select(
    F.col("r.company_registration").alias("registration_number"),
    F.col("r.company_name").alias("company_name"),
    F.col("r.province").alias("province"),
    F.col("r.company_age_days").alias("company_age_days"),
    F.col("r.risk_score").alias("risk_score"),
    F.col("r.risk_category").alias("risk_category"),
    F.col("r.sanctions_hit").alias("is_sanctioned"),
    F.col("s.match_confidence").alias("sanction_match_confidence"),
    F.col("s.matched_sanction_name").alias("matched_sanction_entity"),
    F.col("r.jse_listed").alias("is_jse_listed"),
    F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")).alias("last_updated")
)

# 4. Write to Gold layer
final_view_df.write.format("delta").mode("overwrite").saveAsTable("gold_compliance_dashboard_view")

print(f"\n Gold Master View Complete!")
print(f" Total records in dashboard view: {final_view_df.count()}")
print("Table created: gold_compliance_dashboard_view")

# 5. Display a sample of the final view
print("\n Sample of the final Power BI source table:")
display(final_view_df.limit(10))

StatementMeta(, c008db5c-ed6a-473a-8e86-fda6da862b0f, 3, Finished, Available, Finished, False)

 Creating Gold Master View for Power BI...
 Risk records: 891
 Sanctions records: 56

 Gold Master View Complete!
 Total records in dashboard view: 891
Table created: gold_compliance_dashboard_view

 Sample of the final Power BI source table:


SynapseWidget(Synapse.DataFrame, debb4cf7-2408-43c1-b21c-4f8fece1c9c7)